In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx

In [2]:
def calculate_bic(bn, data):
    """
    Calculates BIC score for a fitted CausalNex BayesianNetwork.
    Formula: BIC = k * ln(n) - 2 * ln(L)
    Returns: bic, log_likelihood
    """
    log_likelihood = 0
    k = 0
    n = len(data)
    
    for node in bn.nodes:
        cpd = bn.cpds[node]
        num_node_states = cpd.shape[0]
        num_parent_configs = cpd.shape[1]
        k += num_parent_configs * (num_node_states - 1)
        parents = list(bn.structure.predecessors(node))
        
        if len(parents) == 0:
            probs = data[node].map(cpd.iloc[:, 0]).values
        else:
            cpd_series = cpd.unstack()
            parent_order = sorted(parents)
            lookup_cols = parent_order + [node]
            lookup_df = data[lookup_cols]
            lookup_keys = [tuple(x) for x in lookup_df.to_numpy()]
            probs = cpd_series.loc[lookup_keys].values
            
        log_likelihood += np.sum(np.log(probs + 1e-9))
    
    bic = k * np.log(n) - 2 * log_likelihood
    
    return bic, log_likelihood

In [3]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import itertools

def add_latent_variable_gmm(bn_baseline, sm_baseline, children_cols, discretised_data, raw_data=None):
    
    if raw_data is None:
        raise ValueError("You must provide the continuous 'raw_data' to accurately estimate the Latent Variable.")
        
    subset_data = raw_data[children_cols].loc[discretised_data.index].copy()
    
    print(f"Estimating LV based on continuous GMM distributions in: {children_cols}")

    scaler = StandardScaler()
    subset_data_scaled = scaler.fit_transform(subset_data)
    
    gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
    proxy_lv = gmm.fit_predict(subset_data_scaled)
    discretised_with_lv = discretised_data.copy()
    discretised_with_lv['LV'] = proxy_lv
    discretised_with_lv['LV'] = discretised_with_lv['LV'].map({0: "State0", 1: "State1"})
    
    sm_with_lv = sm_baseline.copy()
    sm_with_lv.add_node('LV')
    
    for child in children_cols:
        sm_with_lv.add_edge('LV', child)
        
    for u, v in itertools.permutations(children_cols, 2):
        if sm_with_lv.has_edge(u, v):
            print(f"  - Removing direct edge: {u} -> {v}")
            sm_with_lv.remove_edge(u, v)

    bn_with_lv = BayesianNetwork(sm_with_lv)
    bn_with_lv.fit_node_states(discretised_with_lv)
    bn_with_lv.fit_cpds(discretised_with_lv, method="BayesianEstimator", bayes_prior="K2")
    
    bic_lv, ll_lv = calculate_bic(bn_with_lv, discretised_with_lv)
    print(f"LV Model BIC: {bic_lv:.2f} (LogLikelihood: {ll_lv:.2f})")
    
    return bn_with_lv, discretised_with_lv, bic_lv

In [4]:
def discretize_data(df):
    discretised_df = pd.DataFrame(index=df.index)
    TARGET_BINS = 3

    for col in df.columns:
        if df[col].isna().all():
            discretised_df[col] = "Missing"
            continue
        
        no_unique = df[col].nunique()
    
        if no_unique <= 1:
            discretised_df[col] = "Constant"
            continue
    
        if no_unique <= TARGET_BINS:
            discretised_df[col] = df[col].rank(method='dense') - 1
        else:
            try:
                discretised_df[col] = pd.qcut(df[col], q=TARGET_BINS, labels=False, duplicates='drop')
            except (ValueError, TypeError):
                try:
                    discretised_df[col] = pd.cut(df[col], bins=TARGET_BINS, labels=False, include_lowest=True)
                except (ValueError, TypeError):
                    discretised_df[col] = pd.qcut(df[col].rank(method='first'), q=TARGET_BINS, labels=False, duplicates='drop')

        if pd.api.types.is_numeric_dtype(discretised_df[col]):
            valid_values = sorted(discretised_df[col].dropna().unique())
            num_states = len(valid_values)
            
            if num_states == 2:
                label_map = {valid_values[0]: "Low", valid_values[1]: "High"}
            elif num_states == 3:
                label_map = {valid_values[0]: "Low", valid_values[1]: "Medium", valid_values[2]: "High"}
            elif num_states == 4:
                label_map = {valid_values[0]: "VeryLow", valid_values[1]: "Low", valid_values[2]: "High", valid_values[3]: "VeryHigh"}
            else:
                label_map = {val: f"State{int(val)}" for val in valid_values}
            
            discretised_df[col] = discretised_df[col].map(label_map)
            
            discretised_df[col] = discretised_df[col].fillna("Unknown")

    print("Data types after discretization:")
    print(discretised_df.dtypes.value_counts())

    print("\nUnique states per column:")
    print(discretised_df.nunique())

    print("\nDiscretised data preview:")
    print(discretised_df.head())

    if discretised_df.isna().any().any():
        print("\n⚠️ Warning: Some NaN values remain!")
        print(discretised_df.isna().sum()[discretised_df.isna().sum() > 0])
    
    return discretised_df

In [5]:
df_lt = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_bn_no_context.csv")

df_lt = df_lt.drop(["Unnamed: 0"], axis = 1)

df_lt

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,10720.0,29.65,73.0,4.400000,4.970000,8.39,16.482,-13.094653,17.451336,-0.493558,6.55,-11.6,19.1,0.000005,19.195046,45.521991
1,11110.0,29.20,73.0,4.610000,5.210000,13.35,17.824,-11.598256,18.666542,-2.844590,3.61,-9.0,20.9,0.000005,27.187500,70.724103
2,9570.0,25.02,67.0,4.220000,5.280000,29.58,19.798,-1.755188,21.205413,-8.626258,-0.96,-13.2,20.3,0.000004,28.797468,337.935130
3,9810.0,26.21,64.3,3.570000,5.270000,35.74,19.639,-2.029481,20.110710,-7.016844,3.01,-28.9,20.5,0.000005,34.838710,281.231582
4,10670.0,28.35,66.9,3.730000,5.230000,32.59,19.943,-2.706873,18.467399,-5.541350,4.36,-16.2,19.2,0.000005,30.693069,309.501994
5,11280.0,28.20,68.5,4.230000,5.170000,26.68,21.437,0.613211,17.543879,-2.893765,1.59,-9.7,18.6,0.000005,36.454849,339.126224
6,11850.0,27.62,69.9,4.320000,5.020000,21.90,22.689,1.457239,16.735544,-2.417272,1.66,-9.0,20.6,0.000005,39.527027,362.332974
7,12400.0,27.73,71.8,4.190000,4.940000,19.26,23.592,1.930309,16.691874,-1.911556,0.74,-7.2,19.1,0.000004,41.836735,344.608645
8,12860.0,26.96,73.3,4.120000,4.960000,16.30,25.748,-0.986813,17.168991,-1.093729,2.50,-10.7,22.2,0.000005,34.707904,29.635159
9,13360.0,26.00,75.2,4.400000,4.900000,14.45,25.612,0.728400,17.029185,-0.460837,2.74,-12.7,21.9,0.000005,32.986111,115.573864


In [6]:
lt = pd.DataFrame()
lt = df_lt.copy()
lt['EUPC_2'] = lt['EUPC'].shift(2)

In [7]:
lt = lt.dropna()

In [8]:
discretised_lt = discretize_data(lt)

Data types after discretization:
object    17
Name: count, dtype: int64

Unique states per column:
GDPC      3
VABI      3
EMP       3
QOA       3
QOR       3
YUA       3
REST      3
EBSG      3
GGFC      3
LBGD      3
FDI       3
PC        3
ARP       3
CO2C      3
PAM       3
EUPC      3
EUPC_2    3
dtype: int64

Discretised data preview:
  GDPC    VABI  EMP     QOA   QOR   YUA REST    EBSG  GGFC    LBGD     FDI  \
2  Low     Low  Low     Low  High  High  Low     Low  High     Low     Low   
3  Low  Medium  Low     Low  High  High  Low     Low  High     Low  Medium   
4  Low    High  Low     Low  High  High  Low     Low  High     Low    High   
5  Low    High  Low  Medium  High  High  Low  Medium  High     Low     Low   
6  Low    High  Low    High  High  High  Low  Medium   Low  Medium     Low   

       PC     ARP    CO2C   PAM    EUPC  EUPC_2  
2     Low     Low     Low   Low  Medium     Low  
3     Low  Medium     Low  High     Low     Low  
4     Low     Low  Medium   Low  Mediu

In [9]:
sm = StructureModel()

In [10]:
sm.add_edges_from([
    ('EUPC_2', 'FDI')
])

In [11]:
sm.edges

OutEdgeView([('EUPC_2', 'FDI')])

In [12]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [13]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_lt)
bn_baseline.fit_cpds(discretised_lt, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_lt)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUPC_2  -> FDI)
# ==========================================
print("\n--- Testing EUPC_2 -> FDI ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC_2', 'FDI'], 
    discretised_data=discretised_lt,
    raw_data=lt
)

BASELINE BIC: 79.75

--- Testing EUPC_2 -> FDI ---
Estimating LV based on continuous GMM distributions in: ['EUPC_2', 'FDI']
  - Removing direct edge: EUPC_2 -> FDI
LV Model BIC: 93.39 (LogLikelihood: -34.82)


In [14]:
df_lt = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_bn_no_context.csv")

df_lt = df_lt.drop(["Unnamed: 0"], axis = 1)

df_lt

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,10720.0,29.65,73.0,4.400000,4.970000,8.39,16.482,-13.094653,17.451336,-0.493558,6.55,-11.6,19.1,0.000005,19.195046,45.521991
1,11110.0,29.20,73.0,4.610000,5.210000,13.35,17.824,-11.598256,18.666542,-2.844590,3.61,-9.0,20.9,0.000005,27.187500,70.724103
2,9570.0,25.02,67.0,4.220000,5.280000,29.58,19.798,-1.755188,21.205413,-8.626258,-0.96,-13.2,20.3,0.000004,28.797468,337.935130
3,9810.0,26.21,64.3,3.570000,5.270000,35.74,19.639,-2.029481,20.110710,-7.016844,3.01,-28.9,20.5,0.000005,34.838710,281.231582
4,10670.0,28.35,66.9,3.730000,5.230000,32.59,19.943,-2.706873,18.467399,-5.541350,4.36,-16.2,19.2,0.000005,30.693069,309.501994
5,11280.0,28.20,68.5,4.230000,5.170000,26.68,21.437,0.613211,17.543879,-2.893765,1.59,-9.7,18.6,0.000005,36.454849,339.126224
6,11850.0,27.62,69.9,4.320000,5.020000,21.90,22.689,1.457239,16.735544,-2.417272,1.66,-9.0,20.6,0.000005,39.527027,362.332974
7,12400.0,27.73,71.8,4.190000,4.940000,19.26,23.592,1.930309,16.691874,-1.911556,0.74,-7.2,19.1,0.000004,41.836735,344.608645
8,12860.0,26.96,73.3,4.120000,4.960000,16.30,25.748,-0.986813,17.168991,-1.093729,2.50,-10.7,22.2,0.000005,34.707904,29.635159
9,13360.0,26.00,75.2,4.400000,4.900000,14.45,25.612,0.728400,17.029185,-0.460837,2.74,-12.7,21.9,0.000005,32.986111,115.573864


In [15]:
lt = pd.DataFrame()
lt = df_lt.copy()
lt['VABI_2'] = lt['VABI'].shift(2)
lt['QOA_3'] = lt['QOA'].shift(3)
lt['EBSG_3'] = lt['EBSG'].shift(3)
lt['ARP_3'] = lt['ARP'].shift(3)
lt['FDI_1'] = lt['FDI'].shift(1)

In [16]:
lt = lt.dropna()

In [17]:
discretised_lt = discretize_data(lt)

Data types after discretization:
object    21
Name: count, dtype: int64

Unique states per column:
GDPC      3
VABI      3
EMP       3
QOA       3
QOR       3
YUA       3
REST      3
EBSG      3
GGFC      3
LBGD      3
FDI       3
PC        3
ARP       3
CO2C      3
PAM       3
EUPC      3
VABI_2    3
QOA_3     3
EBSG_3    3
ARP_3     3
FDI_1     3
dtype: int64

Discretised data preview:
  GDPC    VABI  EMP     QOA   QOR     YUA REST    EBSG  GGFC    LBGD  ...  \
3  Low  Medium  Low     Low  High    High  Low     Low  High     Low  ...   
4  Low    High  Low     Low  High    High  Low     Low  High     Low  ...   
5  Low    High  Low     Low  High    High  Low  Medium  High     Low  ...   
6  Low    High  Low  Medium  High    High  Low  Medium   Low  Medium  ...   
7  Low    High  Low     Low   Low  Medium  Low    High   Low  Medium  ...   

       PC     ARP    CO2C   PAM    EUPC  VABI_2   QOA_3  EBSG_3   ARP_3  \
3     Low     Low     Low  High  Medium    High  Medium     Low     Low

In [18]:
sm = StructureModel()

In [19]:
sm.add_edges_from([
    ('EUPC', 'PC'),
    ('VABI_2', 'EUPC'),
    ('QOA_3', 'EUPC'),
    ('EBSG_3', 'EUPC'),
    ('FDI_1', 'EUPC'),
    ('ARP_3', 'EUPC'),
    
])

In [20]:
sm.edges

OutEdgeView([('EUPC', 'PC'), ('VABI_2', 'EUPC'), ('QOA_3', 'EUPC'), ('EBSG_3', 'EUPC'), ('FDI_1', 'EUPC'), ('ARP_3', 'EUPC')])

In [21]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [23]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_lt)
bn_baseline.fit_cpds(discretised_lt, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_lt)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUPC  -> PC)
# ==========================================
print("\n--- Testing EUPC -> PC ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC', 'PC'], 
    discretised_data=discretised_lt,
    raw_data=lt
)

# ==========================================
# TEST #2: (VABI_2 -> EUPC)
# ==========================================
print("\n--- Testing VABI_2 -> EUPC ---")

sm_test_2 = sm_baseline.copy()

bn_with_lv_2, data_with_lv_2, bic_lv_2 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_2,
    children_cols=['VABI_2', 'EUPC'], 
    discretised_data=discretised_lt,
    raw_data=lt
)

# ==========================================
# TEST #4: (QOA_3 -> EUPC)
# ==========================================
print("\n--- Testing QOA_3 -> EUPC ---")

sm_test_3 = sm_baseline.copy()

bn_with_lv_3, data_with_lv_3, bic_lv_3 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_3,
    children_cols=['QOA_3', 'EUPC'], 
    discretised_data=discretised_lt,
    raw_data=lt
)

# ==========================================
# TEST #5: (EBSG_3 -> EUPC)
# ==========================================
print("\n--- Testing EBSG_3 -> EUPC ---")

sm_test_4 = sm_baseline.copy()

bn_with_lv_4, data_with_lv_4, bic_lv_4 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_4,
    children_cols=['EBSG_3', 'EUPC'], 
    discretised_data=discretised_lt,
    raw_data=lt
)
# ==========================================
# TEST #6: (FDI_1 -> EUPC)
# ==========================================
print("\n--- Testing FDI_1 -> EUPC ---")

sm_test_5 = sm_baseline.copy()

bn_with_lv_5, data_with_lv_5, bic_lv_5 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_5,
    children_cols=['FDI_1', 'EUPC'], 
    discretised_data=discretised_lt,
    raw_data=lt
)
# ==========================================
# TEST #7: (ARP_3 -> EUPC)
# ==========================================
print("\n--- Testing ARP_3 -> EUPC ---")

sm_test_6 = sm_baseline.copy()

bn_with_lv_6, data_with_lv_6, bic_lv_6 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_6,
    children_cols=['ARP_3', 'EUPC'], 
    discretised_data=discretised_lt,
    raw_data=lt
)



BASELINE BIC: 1469.58

--- Testing EUPC -> PC ---
Estimating LV based on continuous GMM distributions in: ['EUPC', 'PC']
  - Removing direct edge: EUPC -> PC
LV Model BIC: 2725.97 (LogLikelihood: -97.18)

--- Testing VABI_2 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['VABI_2', 'EUPC']
  - Removing direct edge: VABI_2 -> EUPC
LV Model BIC: 1069.02 (LogLikelihood: -94.62)

--- Testing QOA_3 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['QOA_3', 'EUPC']
  - Removing direct edge: QOA_3 -> EUPC
LV Model BIC: 1070.33 (LogLikelihood: -95.27)

--- Testing EBSG_3 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['EBSG_3', 'EUPC']
  - Removing direct edge: EBSG_3 -> EUPC
LV Model BIC: 1069.60 (LogLikelihood: -94.91)

--- Testing FDI_1 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['FDI_1', 'EUPC']
  - Removing direct edge: FDI_1 -> EUPC
LV Model BIC: 1069.60 (LogLikelihood: -94.91)

--- Testing ARP_3 -> EUPC